In [ ]:
from google.colab import drive

drive.mount('/content/drive')

%cd /content/drive/MyDrive/faster_rcnn
%cp VOC2007.zip /content
%cp VOC2012.zip /content
%cd /content

In [ ]:
from pathlib import Path
import zipfile

data_path = Path("data/")
data_path.mkdir(exist_ok=True)

voc2007_zip_path = Path("VOC2007.zip")
voc2012_zip_path = Path("VOC2012.zip")

if not voc2007_zip_path.exists() or not voc2012_zip_path.exists():
    raise RuntimeError("Dataset not found.")

print("Extracting 2007 dataset ...")

with zipfile.ZipFile(voc2007_zip_path, "r") as zip_ref:
    zip_ref.extractall(data_path)

print(f"Extracting 2012 dataset ...")
with zipfile.ZipFile(voc2012_zip_path, "r") as zip_ref:
    zip_ref.extractall(data_path)

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from src.backbone import Backbone
from src.rpn import RPN_Head, RegionProposalNetwork

backbone = Backbone().to(device)
rpn_head = RPN_Head(in_channels=1024, mid_channels=512).to(device)

In [ ]:
from pathlib import Path
import torch

checkpoint_dir = Path("/content/drive/MyDrive/faster_rcnn/checkpoints")
# checkpoint_dir = Path("checkpoints")

existing_checkpoints = sorted(checkpoint_dir.glob("step1_epoch_*.pt"),
                              key = lambda p : int(p.stem.split("_epoch_")[1]))

print(f"Existing checkpoints: {existing_checkpoints}")

if existing_checkpoints:
    latest_checkpoint = existing_checkpoints[-1]
    print(f"Loading checkpoint: {latest_checkpoint}")

    checkpoint = torch.load(latest_checkpoint, map_location=device)
    
    backbone.load_state_dict(checkpoint['backbone_state_dict'])
    rpn_head.load_state_dict(checkpoint['rpn_head_state_dict'])



In [ ]:
from src.backbone import backbone_transform
from src.dataset import get_voc_img_paths_train, get_voc_img_paths_test, create_voc_dataloader

transform = backbone_transform

voc2007_img_paths_train, voc2012_img_paths_train = get_voc_img_paths_train()
voc2007_img_paths_test, voc2012_img_paths_test = get_voc_img_paths_test()

combined_train_img_paths = voc2007_img_paths_train + voc2012_img_paths_train

train_dataloader = create_voc_dataloader(img_paths_list=combined_train_img_paths, transform=transform, batch_size=2, shuffle=True)

In [ ]:
batch_imgs, batch_boxes, batch_labels, img_sizes_before_pad = next(iter(train_dataloader))

rpn_network = RegionProposalNetwork(rpn_head=rpn_head).to(device)

with torch.inference_mode():
    backbone.eval()
    rpn_head.eval()
    rpn_network.eval()

    batch_feature_maps = backbone(batch_imgs.to(device))
    print(f"batch_feature_maps.shape: {batch_feature_maps.shape}")
    batch_scores, batch_proposals = rpn_network(batch_feature_maps, batch_imgs.shape[2], batch_imgs.shape[3], img_sizes_before_pad)
    